# Eval viewer (napari)

Open the per-image TIFFs produced by `evaluate.py` in napari.

## Viewing modes

### Interactive browser (recommended)
Run the **"Interactive browser"** section below to get:
- **2 × 2 grid view**: PCNA / gt / pred / diff each in their own panel with shared zoom/pan.  
  Hide PCNA and diff in the layer list → collapses to side-by-side **gt | pred**.
- **Sample slider** on the right: drag to browse all 865 samples ordered by SSIM  
  (left = best, right = worst). Label shows `file_id`, SSIM, and L1.

### Manual / multi-sample comparison
The original cells at the bottom open individual samples or stack best/median/worst into one viewer using toggleable named layers.

## Prerequisites

Local environment (NOT on cluster — napari needs a display):

```bash
pip install napari[all] pandas tifffile
```

Then download an eval directory from the cluster, e.g.:

```bash
scp -r <user>@<cluster>:<PROJECT_DIR>/runs/<run_name>/eval ./local_runs/<run_name>/
```

In [ ]:
from pathlib import Path

import napari
import numpy as np
import pandas as pd
import tifffile

# Point this at one downloaded eval directory.
EVAL_DIR = Path("../local_runs/test/Cdh1_bc64/eval/test").resolve()
assert EVAL_DIR.exists(), f"Not found: {EVAL_DIR}"

metrics = pd.read_csv(EVAL_DIR / "metrics.csv")
print(f"{len(metrics)} samples in {EVAL_DIR}")
metrics.describe()

## Pick which samples to inspect

The CSV is sorted by `val_ssim` descending. Tail = worst, head = best. Use the table to pick `file_id`s and feed them to the viewer below.

In [ ]:
# Show best / median / worst by val_ssim — a quick sanity sweep.
n = len(metrics)
highlights = pd.concat([
    metrics.head(3).assign(bucket="best"),
    metrics.iloc[n // 2 - 1 : n // 2 + 2].assign(bucket="median"),
    metrics.tail(3).assign(bucket="worst"),
])
highlights

In [ ]:
def load_eval_stack(file_id: str) -> np.ndarray:
    """Read the (4, H, W) eval TIFF for one sample."""
    path = EVAL_DIR / f"{file_id}_eval.tif"
    arr = tifffile.imread(str(path))
    if arr.ndim != 3 or arr.shape[0] != 4:
        raise ValueError(f"Expected (4, H, W), got {arr.shape} for {path}")
    return arr  # channels = [PCNA, pred, gt, diff]


def add_sample_to_viewer(
    viewer: napari.Viewer,
    file_id: str,
    name_prefix: str = "",
) -> None:
    """Add the 4 channels of one sample as 4 napari layers.

    `pred` and `gt` share contrast limits so visual brightness is directly
    comparable. `diff` uses a diverging colormap centered at 0.
    """
    stack = load_eval_stack(file_id)
    pcna, pred, gt, diff = stack[0], stack[1], stack[2], stack[3]

    # Shared contrast limits for pred/gt so they're visually comparable.
    target_lo = float(min(pred.min(), gt.min()))
    target_hi = float(max(pred.max(), gt.max()))
    if target_hi <= target_lo:
        target_hi = target_lo + 1e-6

    # Symmetric contrast for diff so zero is the colormap center.
    diff_abs = float(np.max(np.abs(diff)))
    if diff_abs == 0:
        diff_abs = 1e-6

    prefix = f"{name_prefix}{file_id}_" if name_prefix else f"{file_id}_"

    viewer.add_image(
        pcna,
        name=f"{prefix}PCNA",
        colormap="gray",
        contrast_limits=(float(pcna.min()), float(pcna.max())),
        visible=True,
    )
    viewer.add_image(
        gt,
        name=f"{prefix}gt",
        colormap="viridis",
        contrast_limits=(target_lo, target_hi),
        visible=True,
    )
    viewer.add_image(
        pred,
        name=f"{prefix}pred",
        colormap="viridis",
        contrast_limits=(target_lo, target_hi),
        visible=True,
    )
    viewer.add_image(
        diff,
        name=f"{prefix}diff",
        colormap="bwr",
        contrast_limits=(-diff_abs, diff_abs),
        visible=False,  # off by default; toggle on in the layer panel
    )

## Interactive browser — slider + grid view

Run the two cells below to open a napari viewer with:

- **2 × 2 grid**: PCNA / gt / pred / diff each in their own panel (shared zoom/pan).  
  Hide PCNA and diff in the layer list to collapse to a side-by-side gt | pred view.
- **Sample slider** docked on the right: drag to browse all samples in SSIM order  
  (left = best, right = worst). The label shows `file_id`, SSIM, and L1 for the current frame.

In [ ]:
def update_viewer(viewer: napari.Viewer, idx: int) -> None:
    """Update all 4 layers in-place for sample at position `idx` in metrics."""
    row = metrics.iloc[idx]
    stack = load_eval_stack(row["file_id"])
    pcna, pred, gt, diff = stack[0], stack[1], stack[2], stack[3]

    target_lo = float(min(pred.min(), gt.min()))
    target_hi = float(max(pred.max(), gt.max()))
    if target_hi <= target_lo:
        target_hi = target_lo + 1e-6
    diff_abs = float(np.max(np.abs(diff))) or 1e-6

    viewer.layers["PCNA"].data = pcna
    viewer.layers["PCNA"].contrast_limits = (float(pcna.min()), float(pcna.max()))
    viewer.layers["gt"].data = gt
    viewer.layers["gt"].contrast_limits = (target_lo, target_hi)
    viewer.layers["pred"].data = pred
    viewer.layers["pred"].contrast_limits = (target_lo, target_hi)
    viewer.layers["diff"].data = diff
    viewer.layers["diff"].contrast_limits = (-diff_abs, diff_abs)

In [ ]:
from magicgui.widgets import Container, Label, Slider
from qtpy.QtCore import QTimer

# --- Create viewer with grid mode ---
viewer = napari.Viewer(title=EVAL_DIR.parts[-3])
viewer.grid.enabled = True
viewer.grid.shape = (2, 2)

# Initialise layers with sample 0 (updated in-place by the slider)
stack0 = load_eval_stack(metrics.iloc[0]["file_id"])
pcna0, pred0, gt0, diff0 = stack0[0], stack0[1], stack0[2], stack0[3]
target_lo0 = float(min(pred0.min(), gt0.min()))
target_hi0 = float(max(pred0.max(), gt0.max()))
if target_hi0 <= target_lo0:
    target_hi0 = target_lo0 + 1e-6
diff_abs0 = float(np.max(np.abs(diff0))) or 1e-6

viewer.add_image(pcna0, name="PCNA", colormap="gray",    contrast_limits=(float(pcna0.min()), float(pcna0.max())))
viewer.add_image(gt0,   name="gt",   colormap="viridis", contrast_limits=(target_lo0, target_hi0))
viewer.add_image(pred0, name="pred", colormap="viridis", contrast_limits=(target_lo0, target_hi0))
viewer.add_image(diff0, name="diff", colormap="bwr",     contrast_limits=(-diff_abs0, diff_abs0), visible=False)

# --- Slider widget ---
info_label = Label(value="")

def _on_slide(idx: int) -> None:
    update_viewer(viewer, idx)
    row = metrics.iloc[idx]
    info_label.value = (
        f"{row['file_id']}  |  "
        f"SSIM={row['val_ssim']:.3f}  "
        f"L1={row['val_l1']:.3f}  "
        f"({idx + 1}/{len(metrics)})"
    )

idx_slider = Slider(min=0, max=len(metrics) - 1, value=0, label="Sample")
idx_slider.changed.connect(_on_slide)
_on_slide(0)  # populate label for initial sample

container = Container(widgets=[idx_slider, info_label], layout="vertical")
viewer.window.add_dock_widget(container, area="bottom", name="Sample Browser")

# --- Keyboard navigation (← / →) ---
# QTimer.singleShot(0) defers the focus-return until after the slider's
# spinbox has had a chance to grab focus, so the canvas wins.
def _refocus():
    viewer.window.qt_viewer.canvas.native.setFocus()

@viewer.bind_key("Left")
def prev_sample(v):
    idx_slider.value = max(0, idx_slider.value - 1)
    QTimer.singleShot(0, _refocus)

@viewer.bind_key("Right")
def next_sample(v):
    idx_slider.value = min(len(metrics) - 1, idx_slider.value + 1)
    QTimer.singleShot(0, _refocus)

viewer  # noqa

In [ ]:
# Open one sample. Toggle layers in the napari panel on the left.
file_id = metrics.iloc[0]["file_id"]  # best by val_ssim

viewer = napari.Viewer(title=f"{EVAL_DIR.parts[-3]} / {EVAL_DIR.name}")
add_sample_to_viewer(viewer, file_id)
viewer  # noqa

## Compare multiple samples in one viewer

Each `add_sample_to_viewer` call adds 4 more layers, namespaced by `file_id`. Use the visibility toggles on the left to switch between them.

In [ ]:
# Stack best + median + worst into the same viewer.
for fid in highlights["file_id"]:
    add_sample_to_viewer(viewer, fid)

viewer  # noqa